<a href="https://colab.research.google.com/github/phong6786789/OMNIVOICE-MOD/blob/main/colab_adam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phong Subi - VOICEOMNI MOD

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [4]:
print('Dang cai dat (~1 phut)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cai dat hoan tat!')


Dang cai dat (~1 phut)...
Cai dat hoan tat!


In [5]:
print("🚀 Đang khởi động Phong Subi - OMNIVOICE MOD...")

# ============================================================
# IMPORT
# ============================================================

import os
import re
import time
import shutil
import logging
import requests
import numpy as np
import torch
import gradio as gr


# ============================================================
# PATCH TORCH
# ============================================================

import torch as _torch

if not hasattr(_torch, "_utils"):
    _torch._utils = _torch._C._utils


# ============================================================
# PATCH TRANSFORMERS
# ============================================================

import transformers as _tf


class _SafeAutoFeatureExtractor:

    @staticmethod
    def from_pretrained(model_name, **kwargs):

        try:
            from transformers import AutoConfig

            cfg = AutoConfig.from_pretrained(
                model_name,
                trust_remote_code=True,
                **kwargs
            )

            sr = getattr(
                cfg,
                "sampling_rate",
                24000
            )

        except Exception:
            sr = 24000

        class _Result:
            sampling_rate = sr

        return _Result()


_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor


# ============================================================
# IMPORT OMNIVOICE
# ============================================================

from omnivoice import (
    OmniVoice,
    OmniVoiceGenerationConfig
)

from omnivoice.utils.common import get_best_device


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)


# ============================================================
# THƯ MỤC
# ============================================================

VOICE_DIR = "/content/voices"
CUSTOM_VOICE_DIR = "/content/custom_voices"

os.makedirs(
    VOICE_DIR,
    exist_ok=True
)

os.makedirs(
    CUSTOM_VOICE_DIR,
    exist_ok=True
)


# ============================================================
# GITHUB VOICES
# ============================================================

GITHUB_VOICE_BASE = (
    "https://raw.githubusercontent.com/"
    "phong6786789/"
    "OMNIVOICE-MOD/"
    "main/voices/"
)


# ============================================================
# 27 GIỌNG PRESET
# ============================================================

VOICE_DATA = {

    # ========================================================
    # NỮ - 12 GIỌNG
    # ========================================================

    "♀ Khánh Huyền":
        "khanhhuyentvc_sample.mp3",

    "♀ Thùy Linh":
        "thuylinh_thuyetminh.mp3",

    "♀ Hoài An":
        "vi_female_hoaian_mb.mp3",

    "♀ Hồng Hạnh":
        "vi_female_honghanh_mn_podcast.mp3",

    "♀ Hồng Ngân":
        "vi_female_hongngan_mn_buon.mp3",

    "♀ Khánh Linh":
        "vi_female_khanhlinh_mb.mp3",

    "♀ Kim Phương":
        "vi_female_kimphuong_mb_tKhLy5k.mp3",

    "♀ Nova":
        "vi_female_nova_default.mp3",

    "♀ Thủy Tiên":
        "vi_female_thuytien_mn.mp3",

    "♀ Thùy Trang":
        "vi_female_thuytrang_mb_rzuuQ7F.mp3",

    "♀ Trâm Anh":
        "vi_female_tramanh_mb_sample.mp3",

    "♀ Trần Anh":
        "vi_female_trananh_mb.mp3",


    # ========================================================
    # NAM - 15 GIỌNG
    # ========================================================

    "♂ Đức Trọng":
        "ductrong_sample2.mp3",

    "♂ Đăng Khoa":
        "vi_male_dangkhoa_mb.mp3",

    "♂ Echo":
        "vi_male_echo_default.mp3",

    "♂ Lê Đức":
        "vi_male_leduc_mb_FZBJAUZ.mp3",

    "♂ Lê Hoàng":
        "vi_male_lehoang_mb_SRojRwi.mp3",

    "♂ Lê Nghĩa":
        "vi_male_lenghia_mb_BITWyO7.mp3",

    "♂ Minh Quân":
        "vi_male_minhquan_mb.mp3",

    "♂ Minh Triết":
        "vi_male_minhtriet_mb.mp3",

    "♂ Onyx":
        "vi_male_onyx_default.mp3",

    "♂ Thành Trung":
        "vi_male_thanhtrung_mn_k8K8ONG.mp3",

    "♂ Trí Dũng":
        "vi_male_tridung_mn_sample.mp3",

    "♂ Tuấn Kiệt":
        "vi_male_tuankiet_mn.mp3",

    "♂ Văn Đức":
        "vi_male_vanduc_mn.mp3",

    "♂ Văn Duy":
        "vi_male_vanduy_mb.mp3",

    "♂ Adam":
        "samples_adam.mp3",
}


TOTAL_PRESET_VOICES = len(VOICE_DATA)


# ============================================================
# DOWNLOAD FILE
# ============================================================

def download_file(url, destination):

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    with open(
        destination,
        "wb"
    ) as f:

        f.write(
            response.content
        )


# ============================================================
# TẢI 27 GIỌNG TỪ GITHUB
# ============================================================

print("\n========================================")
print(
    f"⬇️ KIỂM TRA / TẢI "
    f"{TOTAL_PRESET_VOICES} GIỌNG"
)
print("========================================")


VOICE_FILES = {}
available_voices = []
failed_voices = []


for voice_name, filename in VOICE_DATA.items():

    local_path = os.path.join(
        VOICE_DIR,
        filename
    )

    raw_url = (
        GITHUB_VOICE_BASE
        + filename
    )


    # ========================================================
    # FILE ĐÃ CÓ
    # ========================================================

    if os.path.exists(
        local_path
    ):

        print(
            f"✅ {voice_name}"
        )

        VOICE_FILES[
            voice_name
        ] = local_path

        available_voices.append(
            voice_name
        )

        continue


    # ========================================================
    # DOWNLOAD
    # ========================================================

    try:

        print(
            f"⬇️ {voice_name}"
        )

        download_file(
            raw_url,
            local_path
        )


        # Kiểm tra file
        if os.path.getsize(
            local_path
        ) < 1000:

            os.remove(
                local_path
            )

            raise RuntimeError(
                "File tải về không hợp lệ."
            )


        VOICE_FILES[
            voice_name
        ] = local_path


        available_voices.append(
            voice_name
        )


        print(
            "   ✅ Hoàn tất"
        )


    except Exception as e:

        print(
            f"   ❌ Lỗi: {e}"
        )

        failed_voices.append(
            voice_name
        )


# ============================================================
# THÔNG BÁO DOWNLOAD
# ============================================================

print("\n========================================")

print(
    f"✅ Có {len(available_voices)}/"
    f"{TOTAL_PRESET_VOICES} giọng sẵn sàng"
)


if failed_voices:

    print(
        "❌ Không tải được:"
    )

    for voice in failed_voices:

        print(
            "  -",
            voice
        )


print("========================================\n")


# ============================================================
# KIỂM TRA GPU
# ============================================================

print(
    "🔍 Kiểm tra GPU..."
)


for i in range(30):

    if torch.cuda.is_available():

        print(
            "✅ GPU:",
            torch.cuda.get_device_name(0)
        )

        break

    time.sleep(1)

else:

    print(
        "⚠️ Không phát hiện GPU.\n"
        "Google Colab → Runtime → "
        "Change runtime type → T4 GPU"
    )


# ============================================================
# LOAD OMNIVOICE
# ============================================================

DEVICE = get_best_device()


if torch.cuda.is_available():

    MODEL_DTYPE = torch.float16

else:

    MODEL_DTYPE = torch.float32


logger.info(
    f"Loading OmniVoice on {DEVICE}..."
)


model = OmniVoice.from_pretrained(

    "k2-fsa/OmniVoice",

    device_map=DEVICE,

    dtype=MODEL_DTYPE,

    load_asr=True
)


SAMPLING_RATE = model.sampling_rate


logger.info(
    f"✅ OmniVoice ready — "
    f"{SAMPLING_RATE} Hz"
)


# ============================================================
# GENERATION CONFIG
# ============================================================

GEN_CFG = OmniVoiceGenerationConfig(

    num_step=32,

    guidance_scale=1.8,

    denoise=True,

    preprocess_prompt=True,

    postprocess_output=True,

    position_temperature=5.0,

    class_temperature=0.2,

    pad_duration=0.1,

    fade_duration=0.1,
)


# ============================================================
# CACHE
# ============================================================

VOICE_PROMPT_CACHE = {}

CUSTOM_VOICE_FILES = {}

CUSTOM_VOICE_PROMPTS = {}

CUSTOM_VOICE_COUNTER = 0


# ============================================================
# LẤY DANH SÁCH TẤT CẢ GIỌNG
# ============================================================

def get_all_voice_names():

    return (
        list(
            VOICE_FILES.keys()
        )
        +
        list(
            CUSTOM_VOICE_FILES.keys()
        )
    )


# ============================================================
# LẤY FILE GIỌNG
# ============================================================

def get_voice_file(
    voice_name
):

    # Custom voice
    if voice_name in CUSTOM_VOICE_FILES:

        return CUSTOM_VOICE_FILES[
            voice_name
        ]


    # Preset voice
    if voice_name in VOICE_FILES:

        return VOICE_FILES[
            voice_name
        ]


    return None


# ============================================================
# LẤY / TẠO VOICE PROMPT
# ============================================================

def get_voice_prompt(
    voice_name
):

    # ========================================================
    # CUSTOM ĐÃ CACHE
    # ========================================================

    if voice_name in CUSTOM_VOICE_PROMPTS:

        logger.info(
            f"Using custom cache: "
            f"{voice_name}"
        )

        return CUSTOM_VOICE_PROMPTS[
            voice_name
        ]


    # ========================================================
    # PRESET ĐÃ CACHE
    # ========================================================

    if voice_name in VOICE_PROMPT_CACHE:

        logger.info(
            f"Using preset cache: "
            f"{voice_name}"
        )

        return VOICE_PROMPT_CACHE[
            voice_name
        ]


    # ========================================================
    # LẤY FILE
    # ========================================================

    voice_file = get_voice_file(
        voice_name
    )


    if not voice_file:

        raise ValueError(
            "Không tìm thấy giọng."
        )


    if not os.path.exists(
        voice_file
    ):

        raise FileNotFoundError(
            voice_file
        )


    # ========================================================
    # CREATE PROMPT
    # ========================================================

    logger.info(
        f"Creating VoiceClonePrompt: "
        f"{voice_name}"
    )


    voice_prompt = (
        model.create_voice_clone_prompt(
            ref_audio=voice_file
        )
    )


    # ========================================================
    # CACHE
    # ========================================================

    if voice_name in CUSTOM_VOICE_FILES:

        CUSTOM_VOICE_PROMPTS[
            voice_name
        ] = voice_prompt

    else:

        VOICE_PROMPT_CACHE[
            voice_name
        ] = voice_prompt


    logger.info(
        f"✅ Voice cached: "
        f"{voice_name}"
    )


    return voice_prompt


# ============================================================
# NGHE THỬ GIỌNG
# ============================================================

def preview_voice(
    voice_name
):

    path = get_voice_file(
        voice_name
    )


    if not path:

        return None


    if not os.path.exists(
        path
    ):

        return None


    return path


# ============================================================
# NORMALIZE AUDIO
# ============================================================

def normalize_audio(
    audio
):

    # Tensor -> numpy
    if torch.is_tensor(
        audio
    ):

        audio = (
            audio
            .detach()
            .float()
            .cpu()
            .numpy()
        )


    audio = np.asarray(
        audio,
        dtype=np.float32
    )


    audio = np.squeeze(
        audio
    )


    if audio.size == 0:

        return audio


    max_amp = np.max(
        np.abs(
            audio
        )
    )


    if max_amp > 1.0:

        audio = (
            audio
            /
            max_amp
        )


    return audio


# ============================================================
# CLONE GIỌNG NGƯỜI DÙNG
# ============================================================

def create_custom_voice(
    audio_file,
    voice_name
):

    global CUSTOM_VOICE_COUNTER


    # ========================================================
    # CHECK FILE
    # ========================================================

    if not audio_file:

        raise gr.Error(
            "Hãy upload file âm thanh "
            "hoặc thu âm trước."
        )


    if not os.path.exists(
        audio_file
    ):

        raise gr.Error(
            "Không tìm thấy file âm thanh."
        )


    # ========================================================
    # TÊN GIỌNG
    # ========================================================

    if not voice_name:

        voice_name = (
            "Giọng của tôi"
        )


    voice_name = (
        voice_name.strip()
    )


    if not voice_name:

        voice_name = (
            "Giọng của tôi"
        )


    CUSTOM_VOICE_COUNTER += 1


    display_name = (
        f"🎤 {voice_name} "
        f"#{CUSTOM_VOICE_COUNTER}"
    )


    # ========================================================
    # EXTENSION
    # ========================================================

    extension = os.path.splitext(
        audio_file
    )[1]


    if not extension:

        extension = ".wav"


    # ========================================================
    # COPY FILE
    # ========================================================

    destination = os.path.join(

        CUSTOM_VOICE_DIR,

        (
            f"custom_voice_"
            f"{CUSTOM_VOICE_COUNTER}"
            f"{extension}"
        )
    )


    shutil.copy2(
        audio_file,
        destination
    )


    logger.info(
        f"Creating custom voice: "
        f"{display_name}"
    )


    # ========================================================
    # CREATE CLONE PROMPT
    # ========================================================

    try:

        voice_prompt = (
            model.create_voice_clone_prompt(
                ref_audio=destination
            )
        )

    except Exception as e:

        if os.path.exists(
            destination
        ):

            os.remove(
                destination
            )

        raise gr.Error(
            f"Không thể clone giọng:\n{e}"
        )


    # ========================================================
    # SAVE CUSTOM
    # ========================================================

    CUSTOM_VOICE_FILES[
        display_name
    ] = destination


    CUSTOM_VOICE_PROMPTS[
        display_name
    ] = voice_prompt


    # ========================================================
    # UPDATE LIST
    # ========================================================

    choices = get_all_voice_names()


    logger.info(
        f"✅ Custom voice ready: "
        f"{display_name}"
    )


    return (

        gr.Dropdown(
            choices=choices,
            value=display_name
        ),

        destination,

        (
            f"✅ **Đã clone thành công:** "
            f"{display_name}"
        )
    )


# ============================================================
# TẠO TTS
# ============================================================

def generate_voice(
    text,
    voice_name,
    speed,
    pause_duration
):

    # ========================================================
    # CHECK TEXT
    # ========================================================

    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    text = text.strip()


    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    # ========================================================
    # CHECK VOICE
    # ========================================================

    if voice_name not in get_all_voice_names():

        raise gr.Error(
            "Giọng không hợp lệ."
        )


    # ========================================================
    # VOICE PROMPT
    # ========================================================

    try:

        voice_prompt = get_voice_prompt(
            voice_name
        )

    except Exception as e:

        raise gr.Error(
            f"Lỗi xử lý giọng:\n{e}"
        )


    # ========================================================
    # CHIA ĐOẠN
    # 2 LẦN ENTER = 1 ĐOẠN MỚI
    # ========================================================

    paragraphs = [

        p.strip()

        for p in re.split(
            r"\n\s*\n",
            text
        )

        if p.strip()
    ]


    if not paragraphs:

        raise gr.Error(
            "Nội dung không hợp lệ."
        )


    all_audio = []


    # ========================================================
    # GENERATE TỪNG ĐOẠN
    # ========================================================

    for i, paragraph in enumerate(
        paragraphs
    ):

        logger.info(
            f"Generating "
            f"{i + 1}/"
            f"{len(paragraphs)} "
            f"— {voice_name}"
        )


        try:

            result = model.generate(

                text=paragraph,

                voice_clone_prompt=voice_prompt,

                language="vi",

                speed=float(
                    speed
                ),

                generation_config=GEN_CFG
            )


        except Exception as e:

            raise gr.Error(
                f"Lỗi đoạn "
                f"{i + 1}:\n{e}"
            )


        # ====================================================
        # AUDIO
        # ====================================================

        audio = normalize_audio(
            result[0]
        )


        all_audio.append(
            audio
        )


        # ====================================================
        # THÊM KHOẢNG NGHỈ
        # ====================================================

        if i < len(paragraphs) - 1:

            silence_length = int(

                SAMPLING_RATE
                *
                float(
                    pause_duration
                )
            )


            silence = np.zeros(

                silence_length,

                dtype=np.float32
            )


            all_audio.append(
                silence
            )


    # ========================================================
    # NỐI AUDIO
    # ========================================================

    final_audio = np.concatenate(
        all_audio
    )


    final_audio = normalize_audio(
        final_audio
    )


    # ========================================================
    # FLOAT -> INT16
    # ========================================================

    waveform = (

        final_audio
        *
        32767

    ).astype(
        np.int16
    )


    return (
        SAMPLING_RATE,
        waveform
    )


# ============================================================
# GIỌNG MẶC ĐỊNH
# ADAM
# ============================================================

DEFAULT_VOICE = "♂ Adam"


# Nếu Adam lỗi -> Onyx
if DEFAULT_VOICE not in available_voices:

    if "♂ Onyx" in available_voices:

        DEFAULT_VOICE = (
            "♂ Onyx"
        )

    elif available_voices:

        DEFAULT_VOICE = (
            available_voices[0]
        )

    else:

        DEFAULT_VOICE = None


# ============================================================
# DEFAULT PREVIEW
# ============================================================

DEFAULT_PREVIEW = None


if DEFAULT_VOICE:

    DEFAULT_PREVIEW = get_voice_file(
        DEFAULT_VOICE
    )


# ============================================================
# CSS
# GIẢM HIỆN TƯỢNG GIAO DIỆN LẮC / NHẢY
# ============================================================

CSS = """

html {

    overflow-y: scroll !important;
}

body {

    overflow-x: hidden !important;

    margin: 0 !important;
}


/* =========================================
   MAIN CONTAINER
========================================= */

.gradio-container {

    max-width: 820px !important;

    width: 100% !important;

    margin: 0 auto !important;

    padding: 18px !important;

    box-sizing: border-box !important;
}


/* =========================================
   BOX SIZING
========================================= */

.gradio-container * {

    box-sizing: border-box !important;
}


/* =========================================
   INPUT
========================================= */

.gradio-container input,

.gradio-container textarea,

.gradio-container select {

    width: 100% !important;
}


/* =========================================
   TEXTBOX
========================================= */

textarea {

    resize: vertical !important;

    min-height: 180px !important;
}


/* =========================================
   BUTTON
========================================= */

button {

    min-height: 46px !important;
}


/* =========================================
   AUDIO
========================================= */

[data-testid="audio"] {

    min-height: 90px !important;

    width: 100% !important;
}


/* =========================================
   WRAPPER
========================================= */

.gradio-container .wrap {

    width: 100% !important;
}


/* =========================================
   FOOTER
========================================= */

footer {

    display: none !important;
}

"""


# ============================================================
# THEME
# ============================================================

THEME = gr.themes.Soft(
    primary_hue="indigo"
)


# ============================================================
# UI
# ============================================================

print(
    "\n🌐 Khởi động "
    "Phong Subi - OMNIVOICE MOD..."
)


gr.close_all()


with gr.Blocks(

    title="Phong Subi - OMNIVOICE MOD",

    theme=THEME,

    css=CSS,

    fill_width=True

) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        f"""
# 🎙️ Phong Subi - OMNIVOICE MOD

### {TOTAL_PRESET_VOICES} giọng tiếng Việt + Clone giọng riêng

Chọn giọng → nghe thử → nhập nội dung → tạo giọng nói.
"""
    )


    # ========================================================
    # CHỌN GIỌNG
    # ========================================================

    voice_selector = gr.Dropdown(

        choices=get_all_voice_names(),

        value=DEFAULT_VOICE,

        label="🎤 Chọn giọng",

        interactive=True
    )


    # ========================================================
    # NGHE THỬ
    # ========================================================

    preview_audio = gr.Audio(

        value=DEFAULT_PREVIEW,

        label="🔊 Nghe thử giọng",

        interactive=False,

        autoplay=False
    )


    # ========================================================
    # CLONE GIỌNG
    # ========================================================

    with gr.Accordion(

        "➕ Clone giọng của bạn",

        open=False

    ):


        gr.Markdown(
            """
### 🎤 Tạo giọng clone riêng

Bạn có thể **upload file âm thanh** hoặc **thu âm trực tiếp bằng microphone**.

Để clone giọng tốt hơn:

- Chỉ có một người nói
- Không có nhạc nền
- Ít tiếng ồn
- Giọng nói rõ ràng
- Nên sử dụng đoạn âm thanh khoảng 10–30 giây

Chỉ clone giọng của chính bạn hoặc giọng mà bạn được phép sử dụng.
"""
        )


        # ====================================================
        # CUSTOM NAME
        # ====================================================

        custom_voice_name = gr.Textbox(

            label="🏷️ Tên giọng",

            placeholder=(
                "Ví dụ: Giọng của tôi"
            )
        )


        # ====================================================
        # UPLOAD / MICROPHONE
        # ====================================================

        custom_audio = gr.Audio(

            sources=[
                "upload",
                "microphone"
            ],

            type="filepath",

            label=(
                "🎤 Upload hoặc thu âm "
                "giọng mẫu"
            )
        )


        # ====================================================
        # CLONE BUTTON
        # ====================================================

        clone_button = gr.Button(

            "✨ TẠO GIỌNG CLONE",

            variant="secondary"
        )


        # ====================================================
        # STATUS
        # ====================================================

        clone_status = gr.Markdown()


    # ========================================================
    # TEXT INPUT
    # ========================================================

    text_input = gr.Textbox(

        label="📝 Nội dung",

        lines=10,

        placeholder=(
            "Nhập văn bản bạn muốn "
            "chuyển thành giọng nói...\n\n"
            "Xuống dòng 2 lần để tạo "
            "khoảng nghỉ giữa các đoạn."
        )
    )


    # ========================================================
    # SPEED
    # ========================================================

    speed_slider = gr.Slider(

        minimum=0.70,

        maximum=1.30,

        value=0.95,

        step=0.05,

        label="⚡ Tốc độ đọc"
    )


    # ========================================================
    # PAUSE
    # ========================================================

    pause_slider = gr.Slider(

        minimum=0,

        maximum=2.0,

        value=0.3,

        step=0.1,

        label="⏸ Nghỉ giữa đoạn (giây)"
    )


    # ========================================================
    # GENERATE BUTTON
    # ========================================================

    generate_button = gr.Button(

        "🎙️ TẠO GIỌNG NÓI",

        variant="primary"
    )


    # ========================================================
    # OUTPUT AUDIO
    # ========================================================

    output_audio = gr.Audio(

        label="🎧 Kết quả",

        autoplay=False
    )


    # ========================================================
    # EVENT:
    # ĐỔI GIỌNG -> ĐỔI AUDIO NGHE THỬ
    # ========================================================

    voice_selector.change(

        fn=preview_voice,

        inputs=voice_selector,

        outputs=preview_audio,

        show_progress="hidden"
    )


    # ========================================================
    # EVENT:
    # CLONE GIỌNG
    # ========================================================

    clone_button.click(

        fn=create_custom_voice,

        inputs=[

            custom_audio,

            custom_voice_name

        ],

        outputs=[

            voice_selector,

            preview_audio,

            clone_status

        ],

        concurrency_limit=1,

        show_progress="minimal"
    )


    # ========================================================
    # EVENT:
    # GENERATE TTS
    # ========================================================

    generate_button.click(

        fn=generate_voice,

        inputs=[

            text_input,

            voice_selector,

            speed_slider,

            pause_slider

        ],

        outputs=output_audio,

        concurrency_limit=1,

        show_progress="minimal"
    )


# ============================================================
# QUEUE
# ============================================================

demo.queue(
    default_concurrency_limit=1
)


# ============================================================
# LAUNCH
# ============================================================

demo.launch(

    server_name="0.0.0.0",

    share=True,

    show_error=True
)

🚀 Đang khởi động Phong Subi - OMNIVOICE MOD...

⬇️ KIỂM TRA / TẢI 27 GIỌNG
✅ ♀ Khánh Huyền
✅ ♀ Thùy Linh
✅ ♀ Hoài An
✅ ♀ Hồng Hạnh
✅ ♀ Hồng Ngân
✅ ♀ Khánh Linh
✅ ♀ Kim Phương
✅ ♀ Nova
✅ ♀ Thủy Tiên
✅ ♀ Thùy Trang
✅ ♀ Trâm Anh
✅ ♀ Trần Anh
✅ ♂ Đức Trọng
✅ ♂ Đăng Khoa
✅ ♂ Echo
✅ ♂ Lê Đức
✅ ♂ Lê Hoàng
✅ ♂ Lê Nghĩa
⬇️ ♂ Minh Quân
   ✅ Hoàn tất
⬇️ ♂ Minh Triết
   ✅ Hoàn tất
⬇️ ♂ Onyx
   ✅ Hoàn tất
⬇️ ♂ Thành Trung
   ✅ Hoàn tất
⬇️ ♂ Trí Dũng
   ✅ Hoàn tất
⬇️ ♂ Tuấn Kiệt
   ✅ Hoàn tất
⬇️ ♂ Văn Đức
   ✅ Hoàn tất
⬇️ ♂ Văn Duy
   ✅ Hoàn tất
⬇️ ♂ Adam
   ✅ Hoàn tất

✅ Có 27/27 giọng sẵn sàng

🔍 Kiểm tra GPU...
✅ GPU: Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

...

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]


🌐 Khởi động Phong Subi - OMNIVOICE MOD...


/tmp/ipykernel_748/713108088.py:1299: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62bf30d72a03148ba3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
